### 2026-02-06 time-mean biases

AIMIP evaluations of time-mean biases in the train (1979-2014, in-sample) and test (2015-2024, out of sample) periods.

In [1]:
import xarray as xr
import numpy as np
import os
import dataclasses
from dask.diagnostics import ProgressBar
import xesmf
from typing import Literal

In [16]:
DATA_ROOT = '../local_data'
N_REALIZATIONS = 4

@dataclasses.dataclass
class EvaluationVariable:  # mostly these are in case they are missing from the file(s)
    standard_name: str
    short_name: str
    long_name: str
    units: str 

    def validate_metadata(self, ds: xr.Dataset) -> xr.Dataset:
        assert self.short_name in ds.data_vars, "dataset contains variable"
        if "units" not in ds[self.short_name].attrs:
            print(f"Missing units attribute for {self.standard_name}, providing from defaults.")
            ds[self.short_name].attrs["units"] = self.units
        if "long_name" not in ds[self.short_name].attrs:
            print(f"Missing long_name attribute for {self.standard_name}, providing from defaults.")
            ds[self.short_name].attrs["long_name"] = self.long_name
        return ds
        
@dataclasses.dataclass
class ExperimentSubmission:
    submission_dir: str
    file_template: str
    experiment_name: str | None
    label: str | None
    grid: Literal['gr', 'gn']='gn'
    grid_mapping: dict[str, str] | None=None
    renames: dict[str, str] | None=None
    data_root: str=DATA_ROOT

    def __post_init__(self):
        if self.label is None:
            self.label = ''
        if self.experiment_name is None:
            self.experiment_name = ''
        if self.renames is None:
            self.renames = {}
        if self.grid_mapping is None:
            self.grid_mapping = {}
        
    @property
    def name(self) -> str:
        return '/'.join([self.submission_dir, self.experiment_name])

    @property
    def reverse_rename(self) -> dict:
        return {v: k for k, v in self.renames.items()}

    def get_grid_for_variable(self, varname: str) -> str:
        if varname in self.grid_mapping:
            return self.grid_mapping[varname]
        else:
            return self.grid

    def get_variable_path(
        self,
        variable: EvaluationVariable,
        i_r: int,
        table: str,
    ) -> str:
        if variable.short_name in self.renames:
            variable_short_name = self.renames[variable.short_name]
        else:
            variable_short_name = variable.short_name
        file_format = self.file_template.format(
            i_r=i_r,
            table=table,
            varname=variable_short_name,
            grid=self.get_grid_for_variable(variable.short_name),
            label=self.label,
            experiment_name=self.experiment_name,
        )
        return os.path.join(
            self.data_root,
            self.submission_dir,
            self.experiment_name,
            file_format
        )

    def rename_variable(self, ds: xr.Dataset) -> xr.Dataset:
        for data_var in ds.data_vars:
            if data_var in self.renames.values():
                ds = ds.rename({data_var: self.reverse_rename[data_var]})
        return ds

In [17]:
SECONDS_PER_DAY = 86400
TRAINING_DATES = ('1979-10-01', '2014-12-31')
TEST_DATES = ('2015-01-01', '2024-12-31')

FILE_TEMPLATES = {
    'ACE2-1-ERA5': 
    'r{i_r}i1p1f1/{table}/{varname}/{grid}/{label}/{varname}_{table}_ACE2-ERA5_{experiment_name}_r{i_r}i1p1f1_{grid}_197810-202412.nc',
    'ArchesWeather-V2':
    'r{i_r}i1p1f1/{table}/{varname}/{grid}/{varname}_{table}_ArchesWeather_{experiment_name}_r{i_r}i1p1f1_{grid}_197810-202501.nc',
    'ArchesWeatherGen-V2':
    'r{i_r}i1p1f1/{table}/{varname}/{grid}/{varname}_{table}_ArchesWeatherGen_{experiment_name}_r{i_r}i1p1f1_{grid}_197810-202501.nc',
    'NeuralGCM':
    'r{i_r}i1p1f1/{table}/{varname}/{grid}/{label}/{varname}_{table}_NeuralGCM_{experiment_name}_r{i_r}i1p1f1_{grid}_197810-202412.nc',
    'NeuralGCM-HRD':
    'r{i_r}i1p1f1/{table}/{varname}/{grid}/{label}/{varname}_{table}_NeuralGCM-HRD_{experiment_name}_r{i_r}i1p1f1_{grid}_197810-202412.nc',
    'cBottle':
    'r1i1p{i_r}f1/{table}/{varname}/{grid}/{label}/{varname}_{table}_cBottle-1-3_{experiment_name}_r1i1p{i_r}f1_{grid}_197810-202412.nc',
    'MD-1p5':
    'r{i_r}i1p1f1/{table}/{varname}/{grid}/{label}/{varname}_{table}_MD-1p5_{experiment_name}_r{i_r}i1p1f1_{grid}_197810-202412.nc',
    'ERA5_monthly_1deg':
    'mon_1deg/native6_ERA5_an_v1_Amon_{varname}_1978-2024.nc',
}

EXPERIMENT_SUBMISSIONS = [
    ExperimentSubmission(
        submission_dir='Ai2/ACE2-1-ERA5',
        experiment_name='aimip',
        file_template=FILE_TEMPLATES['ACE2-1-ERA5'],
        grid='gr',
        grid_mapping={
            'huss': 'gn',
            'pr': 'gn',
            'ps': 'gn',
            'tas': 'gn',
            'ts': 'gn',
            'uas': 'gn',
            'vas': 'gn',   
        },
        label='v20251130'
    ),
    # ExperimentSubmission(
    #     submission_dir='ArchesWeather/ArchesWeather-V2',
    #     experiment_name='aimip',
    #     file_template=FILE_TEMPLATES['ArchesWeather-V2'],
    #     label=None,
    #     renames={'ts': 'tos'}
    # ),
    # ExperimentSubmission(
    #     submission_dir='ArchesWeather/ArchesWeatherGen-V2',
    #     experiment_name='aimip',
    #     file_template=FILE_TEMPLATES['ArchesWeatherGen-V2'],
    #     label=None,
    #     renames={'ts': 'tos'}
    # ),
    ExperimentSubmission(
        submission_dir='Google/NeuralGCM',
        experiment_name='aimip',
        file_template=FILE_TEMPLATES['NeuralGCM'],
        label='v20251203',
    ),
    # ExperimentSubmission(
    #     submission_dir='Google/NeuralGCM-HRD',
    #     experiment_name='aimip',
    #     file_template=FILE_TEMPLATES['NeuralGCM-HRD'],
    #     label='v20251203',
    # ),
    ExperimentSubmission(
        submission_dir='NVIDIA/CMIP6/AIMIP/NVIDIA/cBottle-1-3',
        experiment_name='aimip',
        file_template=FILE_TEMPLATES['cBottle'],
        label='v20260120',
    ),
    ExperimentSubmission(
        submission_dir='UMD-PARETO/MD-1p5',
        experiment_name='aimip',
        file_template=FILE_TEMPLATES['MD-1p5'],
        label='v20251217',
    ),
    # ExperimentSubmission(
    #     submission_dir='UMD-PARETO/MD-1p5',
    #     experiment_name='aimip',
    #     grid='gr',
    #     file_template=FILE_TEMPLATES['MD-1p5'],
    #     label='v20251217',
    # ),
]

ERA5_1DEG = ExperimentSubmission(
    submission_dir='ERA5',
    file_template=FILE_TEMPLATES['ERA5_monthly_1deg'],
    experiment_name=None,
    label=None,
)

In [18]:
EVALUATION_VARIABLES = [
    EvaluationVariable(
        standard_name='specific_humidity',
        long_name='specific humidity',
        short_name='hus',
        units='-'
    ),
    EvaluationVariable(
        standard_name='surface_specific_humidity',
        long_name='specific humidity at 2 meters',
        short_name='huss',
        units='-'
    ),
    EvaluationVariable(
        standard_name='dew_point_temperature',
        long_name='dew point temperature at 2 meters',
        short_name='tdas',
        units='K'
    ),
    EvaluationVariable(
        standard_name='precipitation_flux',
        long_name='precipitation_flux at the surface',
        short_name='pr',
        units='kg / s / m **2'
    ),
    EvaluationVariable(
        standard_name='surface_air_pressure',
        long_name='air pressure at the surface',
        short_name='ps',
        units='Pa'
    ),
    EvaluationVariable(
        standard_name='air_pressure_at_sea_level',
        long_name='air pressure at mean sea level',
        short_name='psl',
        units='Pa'
    ),
    EvaluationVariable(
        standard_name='air_temperature',
        long_name='air temperature',
        short_name='ta',
        units='K',
    ),
    EvaluationVariable(
        standard_name='air_temperature',
        long_name='air temperature at 2 meters',
        short_name='tas',
        units='K',
    ),
    EvaluationVariable(
        standard_name='surface_temperature',
        long_name='surface temperature',
        short_name='ts',
        units='K',
    ),
    EvaluationVariable(
        standard_name='eastward_wind',
        long_name='eastward wind',
        short_name='ua',
        units='m s-1',
    ),
    EvaluationVariable(
        standard_name='eastward_wind',
        long_name='eastward wind at 10 meters',
        short_name='uas',
        units='m s-1',
    ),
    EvaluationVariable(
        standard_name='northward_wind',
        long_name='northward wind',
        short_name='va',
        units='m s-1',
    ),
    EvaluationVariable(
        standard_name='northward_wind',
        long_name='northward wind at 10 meters',
        short_name='vas',
        units='m s-1',
    ),
    EvaluationVariable(
        standard_name='geopotential_height',
        long_name='geopotential height',
        short_name='zg',
        units='m',
    ),
]

In [19]:
def open_variable_from_path(
    path: str,
    evaluation_variable: EvaluationVariable,
    experiment_submission: ExperimentSubmission
) -> xr.Dataset:
    try:
        variable_dataset = xr.open_dataset(path, chunks={})
    except FileNotFoundError:
        print(f"Not found: {path}")
        variable_dataset = xr.Dataset()
    else:
        print(path)
        variable_dataset = experiment_submission.rename_variable(variable_dataset)
        variable_dataset = evaluation_variable.validate_metadata(variable_dataset)
    # singleton height coordinate for surface variables causes xarray merge conflicts
    variable_dataset = variable_dataset.drop_vars("height", errors="ignore") 
    return variable_dataset

def open_aimip_data(
    experiment_submissions: list[ExperimentSubmission],
    evaluation_variables: list[EvaluationVariable],
    table: Literal['Amon', 'day'],
    n_realizations: int=N_REALIZATIONS,
) -> dict[str, xr.Dataset]:
    experiment_submission_datasets = {}
    for experiment_submission in experiment_submissions:
        print(experiment_submission.name)
        evaluation_variable_datasets = []
        for evaluation_variable in evaluation_variables:
            print(evaluation_variable.standard_name)
            realization_datasets = []
            for i_r in range(1, n_realizations + 1):
                variable_path = experiment_submission.get_variable_path(evaluation_variable, i_r, table)
                realization_dataset = open_variable_from_path(variable_path, evaluation_variable, experiment_submission)
                realization_datasets.append(realization_dataset.expand_dims({'realization': [i_r]}))
            evaluation_variable_dataset = xr.concat(realization_datasets, dim='realization')
            evaluation_variable_datasets.append(evaluation_variable_dataset)
        experiment_submission_dataset = xr.merge(evaluation_variable_datasets, compat='no_conflicts')
        experiment_submission_datasets[experiment_submission.name] =  experiment_submission_dataset
    return experiment_submission_datasets

def time_mean(ds: xr.Dataset, start: str, end: str, time_dim_name: str='time') -> xr.Dataset:
    time_subset = ds.sel(**{time_dim_name: slice(start, end)})
    return time_subset.mean(time_dim_name)

In [20]:
monthly_data = open_aimip_data(
    EXPERIMENT_SUBMISSIONS,
    EVALUATION_VARIABLES,
    table='Amon',
)

Ai2/ACE2-1-ERA5/aimip
specific_humidity
../local_data/Ai2/ACE2-1-ERA5/aimip/r1i1p1f1/Amon/hus/gr/v20251130/hus_Amon_ACE2-ERA5_aimip_r1i1p1f1_gr_197810-202412.nc
../local_data/Ai2/ACE2-1-ERA5/aimip/r2i1p1f1/Amon/hus/gr/v20251130/hus_Amon_ACE2-ERA5_aimip_r2i1p1f1_gr_197810-202412.nc
../local_data/Ai2/ACE2-1-ERA5/aimip/r3i1p1f1/Amon/hus/gr/v20251130/hus_Amon_ACE2-ERA5_aimip_r3i1p1f1_gr_197810-202412.nc
../local_data/Ai2/ACE2-1-ERA5/aimip/r4i1p1f1/Amon/hus/gr/v20251130/hus_Amon_ACE2-ERA5_aimip_r4i1p1f1_gr_197810-202412.nc
surface_specific_humidity
../local_data/Ai2/ACE2-1-ERA5/aimip/r1i1p1f1/Amon/huss/gn/v20251130/huss_Amon_ACE2-ERA5_aimip_r1i1p1f1_gn_197810-202412.nc
../local_data/Ai2/ACE2-1-ERA5/aimip/r2i1p1f1/Amon/huss/gn/v20251130/huss_Amon_ACE2-ERA5_aimip_r2i1p1f1_gn_197810-202412.nc
../local_data/Ai2/ACE2-1-ERA5/aimip/r3i1p1f1/Amon/huss/gn/v20251130/huss_Amon_ACE2-ERA5_aimip_r3i1p1f1_gn_197810-202412.nc
../local_data/Ai2/ACE2-1-ERA5/aimip/r4i1p1f1/Amon/huss/gn/v20251130/huss_Amon_ACE

In [21]:
era5_monthly_1deg = open_aimip_data(
    [ERA5_1DEG],
    EVALUATION_VARIABLES,
    table='Amon',
    n_realizations=1,
)
era5_monthly_1deg = (
    era5_monthly_1deg['ERA5/']
    .isel(realization=0).squeeze()
    .drop_vars(['forecast_period', 'originating_centre'])
    .sel(lat=slice(-85, 85))
)

ERA5/
specific_humidity
../local_data/ERA5/mon_1deg/native6_ERA5_an_v1_Amon_hus_1978-2024.nc
surface_specific_humidity
Not found: ../local_data/ERA5/mon_1deg/native6_ERA5_an_v1_Amon_huss_1978-2024.nc
dew_point_temperature
Not found: ../local_data/ERA5/mon_1deg/native6_ERA5_an_v1_Amon_tdas_1978-2024.nc
precipitation_flux
Not found: ../local_data/ERA5/mon_1deg/native6_ERA5_an_v1_Amon_pr_1978-2024.nc
surface_air_pressure
../local_data/ERA5/mon_1deg/native6_ERA5_an_v1_Amon_ps_1978-2024.nc
air_pressure_at_sea_level
Not found: ../local_data/ERA5/mon_1deg/native6_ERA5_an_v1_Amon_psl_1978-2024.nc
air_temperature
../local_data/ERA5/mon_1deg/native6_ERA5_an_v1_Amon_ta_1978-2024.nc
air_temperature
../local_data/ERA5/mon_1deg/native6_ERA5_an_v1_Amon_tas_1978-2024.nc
surface_temperature
../local_data/ERA5/mon_1deg/native6_ERA5_an_v1_Amon_ts_1978-2024.nc
eastward_wind
../local_data/ERA5/mon_1deg/native6_ERA5_an_v1_Amon_ua_1978-2024.nc
eastward_wind
../local_data/ERA5/mon_1deg/native6_ERA5_an_v1_Amon

/tmp/ipykernel_160477/3527488871.py:7: FutureWarning: In a future version, xarray will not decode the variable 'forecast_period' into a timedelta64 dtype based on the presence of a timedelta-like 'units' attribute by default. Instead it will rely on the presence of a timedelta64 'dtype' attribute, which is now xarray's default way of encoding timedelta64 values.
To continue decoding into a timedelta64 dtype, either set `decode_timedelta=True` when opening this dataset, or add the attribute `dtype='timedelta64[ns]'` to this variable on disk.
To opt-in to future behavior, set `decode_timedelta=False`.
  variable_dataset = xr.open_dataset(path, chunks={})
/tmp/ipykernel_160477/3527488871.py:7: FutureWarning: In a future version, xarray will not decode the variable 'forecast_period' into a timedelta64 dtype based on the presence of a timedelta-like 'units' attribute by default. Instead it will rely on the presence of a timedelta64 'dtype' attribute, which is now xarray's default way of enc

In [22]:
training_time_mean = {
    k: time_mean(v, start=TRAINING_DATES[0], end=TRAINING_DATES[1])
    for k, v in monthly_data.items()
}

test_time_mean = {
    k: time_mean(v, start=TEST_DATES[0], end=TEST_DATES[1])
    for k, v in monthly_data.items()
}

KeyError: "'time' is not a valid dimension or coordinate for Dataset with dimensions FrozenMappingWarningOnValuesAccess({'realization': 4})"

In [ ]:
with ProgressBar():
    training_time_mean = {
        k: v.compute() for k, v in training_time_mean.items()
    }
    test_time_mean = {
        k: v.compute() for k, v in test_time_mean.items()
    }

In [ ]:
training_time_mean_era5 = time_mean(
    era5_monthly_1deg, start=TRAINING_DATES[0], end=TRAINING_DATES[1]
)
test_time_mean_era5 = time_mean(
    era5_monthly_1deg, start=TEST_DATES[0], end=TEST_DATES[1]
)

In [ ]:
with ProgressBar():
    training_time_mean_era5 = training_time_mean_era5.compute()
    test_time_mean_era5 = test_time_mean_era5.compute()

In [ ]:
regridded_training_time_means = []
regridded_test_time_means = []
for experiment_submission_name, dataset in training_time_mean.items():
    print(experiment_submission_name)
    regridder = xesmf.Regridder(
        dataset.isel(realization=0).squeeze(),
        training_time_mean_era5,
        method='conservative'
    )
    regridded_training_time_mean = regridder.regrid_dataset(dataset)
    regridded_test_time_mean = regridder.regrid_dataset(test_time_mean[experiment_submission_name])
    regridded_training_time_means.append(
        regridded_training_time_mean.expand_dims(
            {
                'experiment_submission': [experiment_submission_name]
            }
        )
    )
    regridded_test_time_means.append(
        regridded_test_time_mean.expand_dims(
            {
                'experiment_submission': [experiment_submission_name]
            }
        )
    )
regridded_training_time_means = xr.concat(regridded_training_time_means, dim='experiment_submission', join='inner')
regridded_test_time_means = xr.concat(regridded_test_time_means, dim='experiment_submission', join='inner')

In [ ]:
training_biases = regridded_training_time_means - training_time_mean_era5
test_biases = regridded_test_time_means - test_time_mean_era5

In [ ]:
# test_biases

In [ ]:
# training_biases

In [ ]:
training_biases.ta.isel(plev=-2).plot(row='experiment_submission', col='realization', cmap='RdBu_r')

In [ ]:
test_biases.ta.isel(plev=-2).plot(row='experiment_submission', col='realization', cmap='RdBu_r')

In [ ]:
weights = np.cos(np.deg2rad(training_biases.lat))
training_rmsb = np.sqrt((training_biases ** 2).weighted(weights).mean(dim=['lat', 'lon']))
test_rmsb = np.sqrt((test_biases ** 2).weighted(weights).mean(dim=['lat', 'lon']))

In [ ]:
training_rmsb.ta.to_dataframe().unstack().T.plot.bar()

In [ ]:
test_rmsb.ta.to_dataframe().unstack().T.plot.bar()

In [ ]:
training_rmsb[['ta']].to_stacked_array(
    name='variable_stats',
    new_dim='variable_name',
    sample_dims=['experiment_submission', 'realization']
).mean('realization').to_dataframe().unstack()['variable_stats'].T.plot.bar()

In [ ]:
test_rmsb[['ta']].to_stacked_array(
    name='variable_stats',
    new_dim='variable_name',
    sample_dims=['experiment_submission', 'realization']
).mean('realization').to_dataframe().unstack()['variable_stats'].T.plot.bar()